In [ ]:
import os, pickle
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
from py_util_dx.py_utils import setProjectPath

In [ ]:
mpl.rcParams['font.family'] = 'helvetica'   # Global font family
mpl.rcParams['font.size'] = 12              # Global font size 
full_width = 8.5
half_width = full_width/2

In [ ]:
projectPath, mainResultsPath = setProjectPath()

dataset_name = 'IBC'      # MDTB, Nishimoto, HCPur100, Language, Demand, IBC

surface_helpers_dir = os.path.join(projectPath, 'surface_helpers')
resultsPath = os.path.join(mainResultsPath, 'START_A2_smooth_decomposition', dataset_name)

PKL_output = os.path.join(resultsPath, 'output.pkl')
with open(PKL_output, 'rb') as pf:
    output = pickle.load(pf)

In [ ]:
PKL_vis_output = os.path.join(resultsPath, 'vis_output.pkl')
vis_output = {}

In [ ]:
smooth_kernels = list(output.keys())
ROIs = list(output['orig_mm'].keys())

v_s_over_gs = {}

for roi in ROIs:
    v_s_over_gs[roi] = []
    
    for smooth_kernel in smooth_kernels:
        v_s_over_gs[roi].append(output[smooth_kernel][roi][0, 1] / (output[smooth_kernel][roi][0, 0] + output[smooth_kernel][roi][0, 1]))

    v_s_over_gs[roi] = np.array(v_s_over_gs[roi])

vis_output['v_s_over_gs'] = v_s_over_gs

In [ ]:
v_s_over_gs

In [ ]:
colors = [(18/255, 75/255, 141/255), (0, 230/255, 230/255), (255/255, 165/255, 0/255), (240/255, 10/255, 240/255)]

fig, ax = plt.subplots()
fig.set_figwidth(half_width) 
fig.set_figheight(4.5) 

for roiI in np.arange(len(ROIs)):
    ax.scatter(0, v_s_over_gs[ROIs[roiI]][0], color=colors[roiI], marker='o')
    ax.plot(np.arange(1, len(smooth_kernels)), v_s_over_gs[ROIs[roiI]][1:], color=colors[roiI], linestyle='-', label=ROIs[roiI])

ax.hlines(0, xmin=0, xmax=len(smooth_kernels)-1, linestyles='--', colors=(150/255, 150/255, 150/255))
ax.hlines(1, xmin=0, xmax=len(smooth_kernels)-1, linestyles='--', colors=(150/255, 150/255, 150/255))
ax.text(0, 0.05, 'group')
ax.text(0, 0.95, 'individual')

# xlabels = [l.strip('_mm') for l in smooth_kernels]
xlabels = ['overall', '>10 mm', '8-10 mm', '6-8 mm', '4-6 mm', '2-4 mm', '0-2 mm']

ax.set_xticks(np.arange(len(smooth_kernels)), xlabels, rotation=45)
ax.set_title('$V_s/(V_s+V_g)$', fontsize=12)

ax.legend(frameon=False, loc='right')
ax.set_ylim([-0.05, 1.05])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(resultsPath, f'vs_over_gs.jpg'), dpi=500, format='jpeg')

with open(PKL_vis_output, 'wb') as pk:
    pickle.dump(vis_output, pk)

plt.show()